# Gaussian beam tests

We simulate a simple Gaussian source as a cross-check.

Sanity check the PyPO installation

In [ ]:
# Sanity check installation of PyPO
import PyPO
import os, datetime

pypo_file = PyPO.__file__
pypo_dir = os.path.dirname(pypo_file)
print(pypo_dir)
fp_time = datetime.datetime.fromtimestamp(os.path.getmtime(pypo_file))
print(f"Installed {fp_time}")


print("Shared library installed times")
pypo_files = os.listdir(pypo_dir)
compiled_files = []
for f in pypo_files:
    if f.endswith('.so'):
        compiled_files.append(f)
        fp = os.path.join(pypo_dir, f)
        fp_time = datetime.datetime.fromtimestamp(os.path.getmtime(fp))
        print(f"{f:18s} : {fp_time}")
    
print()
print("Shared library build times")
build_dir = "/home/pgrimes/github/PyPO/build/lib.linux-x86_64-cpython-314/PyPO"
compiled_files = []
for f in pypo_files:
    if f.endswith('.so'):
        compiled_files.append(f)
        fp = os.path.join(build_dir, f)
        fp_time = datetime.datetime.fromtimestamp(os.path.getmtime(fp))
        print(f"{f:18s} : {fp_time}")
        

In [ ]:
%matplotlib widget

import numpy as np
import scipy as sp
import matplotlib.pyplot as plt

from PyPO.System import System
from PyPO.Enums import FieldComponents, CurrentComponents, Units, Scales


# Create the test system

We create a Gaussian beam source at the origin, pointing in the positive Z direction.

In [ ]:
s = System()

## Calculate the Gaussian beam properties

We set up a 300 GHz Gaussian beam source with a 5λ beamwaist, normalized to 4π Watts in emitted power (for compatibility with GRASP, which normalizes sources to 4π Watts by default).  This choice of emitted power means that farfield grids should have units in dBi.

The source is defined on a grid $5 w_0$ in diameter.  From the Gaussian beam formulae, we calculate the field normalization and farfield $1/e$ divergence angle and FWHM angle.

In [ ]:
# Setting up parameters for defining simulation.
# Source parameters and distances
lam = 1*Units.MM                # Wavelength of light in mm

w0 = 5*lam  # Gaussian beamwaist

r_source = 2.5*w0               # Radius of disk in mm
print(f"Radius of source grid     : {r_source:7.3f} mm")

E0 = np.sqrt(8/(w0**2)) # Normalise to 4pi Watts
print(f"Field normalization       : {E0:7.3g} √Watt")

theta_0 = np.rad2deg(np.atan(lam/(np.pi*w0))) # Calculate far-field divergence angle
print(f"Farfield divergence angle : {theta_0:7.3f}°")

fwhm = 1.18*theta_0
print(f"Farfield FWHM angle       : {fwhm:7.3f}°")

Next, we set up our output grids.

We will evaluate the field due to the source on a far-field grid $5 \theta_0$ in diameter, and on a near-field grid $5w$ in diameter at a distance of 20 $w_0$ along the z axis.

The Gaussian beam width at $z=20 w_0$ is given by:

$$ w(z) = w_0 \sqrt{1 + \bigg(\frac{\lambda z}{\pi w_0^2}\bigg)^2} $$

In [ ]:
z_grid = 20*w0
print(f"Near-field grid distance     : {z_grid:7.3f} mm")

w_grid = w0*np.sqrt(1 + ((lam*z_grid)/(np.pi*w0**2))**2)
print(f"Beamwidth at near-field grid : {w_grid:7.3f} mm")

## Grid size for convergence

We now calculate the size of the grid on the source required for the PO surface integrals over our test angles.

The GRASP manual offers the following guidelines for when the PO surface integral will converge:

For focused reflectors of diameter D on a polar grid:


| Axis | Points |
|------|--------|
| Radial | $ \mathrm{int}(z/2.4)$ |
| Angular | $ \mathrm{int}(z) $ |

where 

$$ z = 1.09 \pi \frac{D}{\lambda} \sin \theta_0 + 10 $$

and $\theta_0$ is the maximum angle from the beam direction at which the integral shall converge. For our test case, $\theta_0$ is the angle from the source to the edge of the farfield grid, or the edge of the nearfield grid.

The maximum required size of the grid is given for 90° incidence angles when calculating the backscattered field. For this source and target location, the integrand varies by two cycles per wavelength of distance, and thus integration points must be spaced a maximum of $\lambda/2$ apart.

For a polar grid, the spacing $s$ of $n$ angular points at the rim is given by $s = \pi D / n$.  Thus the maximum number of angular points required is:

| Axis | Points |
|------|--------|
| Radial | $ \mathrm{int}(D / 4 \lambda) $ |
| Angular | $ \mathrm{int}(2 \pi D / \lambda) $ |

In [ ]:
## Calculate the required source grid size for the far-field case

# This estimate is good for about -30-40 dB field accuracy
z = 1.09*np.pi*2*r_source/lam*np.sin(np.deg2rad(theta_0*5)) + 10

po1 = int(z/2.4)
po2 = int(z)

max_po1 = int(2*r_source/lam)
max_po2 = int(4*np.pi*r_source/lam)

print(f"Far-Field Case:")
print(f"Estimated Grid size : ({po1:d}, {po2:d})")
print(f"Maximum Grid size   : ({max_po1:d}, {max_po2:d})")

In [ ]:
## Calculate the required source grid size for the near-field case

# Calculate the angle from one side of the source to the opposite side of the near-field grid
theta = np.atan( (5*w_grid + r_source)  / z_grid)

# This estimate is good for about -30-40 dB field accuracy
z = 1.09*np.pi*2*r_source/lam*np.sin(theta) + 10

po1 = int(z/2.4)
po2 = int(z)

max_po1 = int(2*r_source/lam)
max_po2 = int(4*np.pi*r_source/lam)

print(f"Near-Field Case:")
print(f"Estimated Grid size : ({po1:d}, {po2:d})")
print(f"Maximum Grid size   : ({max_po1:d}, {max_po2:d})")

From this we can see that the near-field case is the limiting one, and also that the maximum grid size is modest.  As such, we might as well use the maxmium grid size.

## Create the source

We now create the source plane grid and populate with a Gaussian electric field polarized in the x direction, along with the associated magnetic current polarized along the y direction.

In [ ]:
# Setting up surface dictionaries and source current distributions.
source_plane = {
        "name"      : "source_plane",
        "gmode"     : "uv",
        # "lims_x"    : np.array([-r_disk, r_disk]),
        # "lims_y"    : np.array([-r_disk, r_disk]),
        "lims_u"    : np.array([0, r_source]),
        "lims_v"    : np.array([0, 360])*Units.DEG,
        "gridsize"  : np.array([max_po1, max_po2])
        }

s.addPlane(source_plane)

source_grid = s.generateGrids("source_plane")

GPODict = {
        "name"  : "source",
        "lam"   : lam,
        "w0x"   : w0,
        "w0y"   : w0,
        "n"     : 1.0,
        "dxyx"  : 0.0,
        "E0"    : E0,
        "pol"   : np.array([1, 0, 0])
}

s.createGaussian(GPODict, "source_plane")

Let's examine the source.

In [ ]:
s.plotBeam2D("source", FieldComponents.Ex, norm=False)
s.plotBeam2D("source", CurrentComponents.My, norm=False)


We also create a scalar version of the source E-field x component, to test the scalar field propagation

In [ ]:
source_field = 'source'

s.scalarfields[f"scalar_{source_field}"] = PyPO.PyPOTypes.scalarfield(s.fields[source_field][FieldComponents.Ex.value])
s.scalarfields[f"scalar_{source_field}"].setMeta(s.fields[source_field].surf, s.fields[source_field].k)

In [ ]:
s.plotBeam2D('scalar_source', comp=FieldComponents.NONE, norm=False)

## Create the far- and near-field grids

We now create the far-field and near-field grids.

To get good resolution in the near and far-field grids of $5 w_0$ and $5 \theta_0$ diameter, we use ~200 points in the linear dimensions, and 360 points in the angular direction.

In [ ]:
ff_size = 2.5*theta_0
n_grid = 201

farfield = {
            "name"      : "farfield",
            "gmode"     : "AoE",
            "lims_Az"    : np.array([-ff_size, ff_size])*Units.DEG,
            "lims_El"    : np.array([-ff_size, ff_size])*Units.DEG,
            "gridsize"  : np.array([n_grid, n_grid])
            }

s.addPlane(farfield)

In [ ]:
nf_radius = w_grid*2.5

nearfield = {
            "name"      : "nearfield",
            "gmode"     : "uv",
            "lims_u"    : np.array([0, nf_radius]),
            "lims_v"    : np.array([0, 360])*Units.DEG,
            "gridsize"  : np.array([n_grid, 360])
            }

s.addPlane(nearfield)
s.translateGrids('nearfield', np.array([0, 0, z]))

# Calculate farfield

We now set up the PO calculation required to calculate the farfield

In [ ]:
po_to_ff = {
        "t_name"    : "farfield",
        "s_current" : "source",
        "mode"      : "FF",
        "name_EH"   : "EH_ff",
        "device"    : "GPU"
        }

s.runPO(po_to_ff)

# Examine farfield results

In [ ]:
vmax = 20*np.log10(np.max(np.abs(s.fields['EH_ff'].Ex)))
vmin = vmax -40

s.plotBeam2D("EH_ff", FieldComponents.Ex, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_ff", FieldComponents.Ey, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_ff", FieldComponents.Ez, norm=False, vmin=vmin, vmax=vmax)

In [ ]:
vmax = 20*np.log10(np.max(np.abs(s.fields['EH_ff'].Hy)))
vmin = vmax -80

s.plotBeam2D("EH_ff", FieldComponents.Hx, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_ff", FieldComponents.Hy, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_ff", FieldComponents.Hz, norm=False, vmin=vmin, vmax=vmax)

In [ ]:
grid = s.generateGrids("farfield")

In [ ]:
field = 'EH_ff'

row = int(n_grid/2)

vmax = 20*np.log10(np.max(np.abs(s.fields[field].Ex)))
vmin = vmax -40

fig, ax = plt.subplots(1,2, figsize=(10,5), gridspec_kw={'wspace':0.5})
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Ex[:,row])**2), label='$E_x$')
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Ey[:,row])**2), label='$E_y$')
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Ez[:,row])**2), label='$E_z$')
ax[0].axvline(fwhm/2, ls=":")
ax[0].axhline(vmax-3, ls=":")
ax[0].set_ylim((vmin, vmax+1))
ax[0].legend()
ax[0].set_xlabel('Angle (°)')
ax[0].set_ylabel('E field power (dBi)')

ax[1].plot(np.rad2deg(grid.x[:,row]), np.unwrap(np.angle(s.fields[field].Ex[:,row])))
ax[1].set_xlabel('Angle (°)')
ax[1].set_ylabel('Phase (rad)')

In [ ]:
field = 'EH_ff'

row = int(n_grid/2)

vmax = 20*np.log10(np.max(np.abs(s.fields[field].Hy)))
vmin = vmax -40

fig, ax = plt.subplots(1,2, figsize=(10,5), gridspec_kw={'wspace':0.5})
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Hx[:,row])**2), label='$H_x$')
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Hy[:,row])**2), label='$H_y$')
ax[0].plot(np.rad2deg(grid.x[:,row]), 10*np.log10(np.abs(s.fields[field].Hz[:,row])**2), label='$H_z$')
ax[0].axvline(fwhm/2, ls=":")
ax[0].axhline(vmax-3, ls=":")
ax[0].set_ylim((vmin, vmax+1))
ax[0].legend()
ax[0].set_xlabel('Angle (°)')
ax[0].set_ylabel('H field power (dBi)')

ax[1].plot(np.rad2deg(grid.x[:,row]), np.unwrap(np.angle(s.fields[field].Hy[:,row])))
ax[1].set_xlabel('Angle (°)')
ax[1].set_ylabel('Phase (rad)')

# Calculate nearfield

In [ ]:
po_to_nf = {
        "t_name"    : "nearfield",
        "s_current" : "source",
        "mode"      : "EH",
        "name_EH"   : "EH_nf",
        "device"    : "GPU"
        }

s.runPO(po_to_nf)

In [ ]:
vmax = 20*np.log10(np.max(np.abs(s.fields['EH_nf'].Ex)))
vmin = vmax -80

s.plotBeam2D("EH_nf", FieldComponents.Ex, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_nf", FieldComponents.Ey, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_nf", FieldComponents.Ez, norm=False, vmin=vmin, vmax=vmax)

In [ ]:
vmax = 20*np.log10(np.max(np.abs(s.fields['EH_nf'].Hy)))
vmin = vmax -80

s.plotBeam2D("EH_nf", FieldComponents.Hx, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_nf", FieldComponents.Hy, norm=False, vmin=vmin, vmax=vmax)
s.plotBeam2D("EH_nf", FieldComponents.Hz, norm=False, vmin=vmin, vmax=vmax)

In [ ]:
20*np.log10(np.exp(-1))

In [ ]:
field = 'EH_nf'
grid = s.generateGrids(s.fields[field].surf)

row = 0

xlabel = 'Radius (mm)'
vmax = 20*np.log10(np.max(np.abs(s.fields[field].Ex)))
vmin = vmax -40

fig, ax = plt.subplots(1,2, figsize=(10,5), gridspec_kw={'wspace':0.5})
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Ex[:,row])**2), label='$E_x$')
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Ey[:,row])**2), label='$E_y$')
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Ez[:,row])**2), label='$E_z$')
ax[0].axvline(w_grid, ls=":")
ax[0].axhline(vmax+20*np.log10(np.exp(-2)), ls=":")
ax[0].set_ylim((vmin, vmax+1))
ax[0].legend()
ax[0].set_xlabel(xlabel)
ax[0].set_ylabel('E field power (dBi)')

ax[1].plot(grid.x[:,row], np.unwrap(np.angle(s.fields[field].Ex[:,row])))
ax[1].set_xlabel(xlabel)
ax[1].set_ylabel('E field Phase (rad)')

In [ ]:
field = 'EH_nf'
grid = s.generateGrids(s.fields[field].surf)

row = 0

xlabel = 'Radius (mm)'
vmax = 20*np.log10(np.max(np.abs(s.fields[field].Hy)))
vmin = vmax -40

fig, ax = plt.subplots(1,2, figsize=(10,5), gridspec_kw={'wspace':0.5})
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Hx[:,row])**2), label='$H_x$')
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Hy[:,row])**2), label='$H_y$')
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.fields[field].Hz[:,row])**2), label='$H_z$')
ax[0].axvline(w_grid, ls=":")
ax[0].axhline(vmax+20*np.log10(np.exp(-2)), ls=":")
ax[0].set_ylim((vmin, vmax+1))
ax[0].legend()
ax[0].set_xlabel(xlabel)
ax[0].set_ylabel('H-field power (dBi)')

ax[1].plot(grid.x[:,row], np.unwrap(np.angle(s.fields[field].Ex[:,row])))
ax[1].set_xlabel(xlabel)
ax[1].set_ylabel('H-field Phase (rad)')

# Nearfield Scalar calculation

We now repeat the nearfield calculations with the scalar field.

In [ ]:
nf_radius = w_grid*2.5

n_grid = 101

test_nearfield = {
            "name"      : "test_nearfield",
            "gmode"     : "uv",
            "lims_u"    : np.array([0, nf_radius]),
            "lims_v"    : np.array([0, 360])*Units.DEG,
            "gridsize"  : np.array([n_grid, int(n_grid*2*np.pi)])
            }

s.addPlane(test_nearfield)
s.translateGrids('test_nearfield', np.array([0, 0, z]))

po_to_nf = {
        "t_name"        : "test_nearfield",
        "s_scalarfield" : "scalar_source",
        "mode"          : "scalar",
        "name_field"    : "scalar_nf",
        "device"        : "GPU"
        }

s.runPO(po_to_nf)

In [ ]:
s.scalarfields['scalar_source'].S[1, 144]

In [ ]:
sfield = "scalar_nf"

vmax = 20*np.log10(np.max(np.abs(s.scalarfields[sfield].S)))
vmin = vmax -80

s.plotBeam2D(sfield, FieldComponents.NONE, norm=False, vmin=vmin, vmax=vmax)

In [ ]:
grid = s.generateGrids(s.scalarfields[sfield].surf)

row = 0

xlabel = 'Radius (mm)'
vmax = 20*np.log10(np.max(np.abs(s.scalarfields[sfield].S)))
vmin = vmax -40

fig, ax = plt.subplots(1,2, figsize=(10,5), gridspec_kw={'wspace':0.5})
ax[0].plot(grid.x[:,row], 10*np.log10(np.abs(s.scalarfields[sfield].S[:,row])**2))
ax[0].axvline(w_grid, ls=":")
ax[0].axhline(vmax+20*np.log10(np.exp(-2)), ls=":")
ax[0].set_ylim((vmin, vmax+1))
ax[0].set_xlabel(xlabel)
ax[0].set_ylabel('Scalar field power (dBi)')

ax[1].plot(grid.x[:,row], np.unwrap(np.angle(s.scalarfields[sfield].S[:,row])))
ax[1].set_xlabel(xlabel)
ax[1].set_ylabel('Scalar field Phase (rad)')